# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   str    
 1   continent  1704 non-null   str    
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   str    
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), str(3)
memory usage: 135.3 KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: <ArrowStringArray>
['Asia', 'Europe', 'Africa', 'Americas', 'Oceania']
Length: 5, dtype: str
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [3]:

# Task 1 - Scatter: Life Expectancy Change Over Time

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Select Asia and the required years
asia = df.loc[
    (df['continent'] == 'Asia') &
    (df['year'].isin([2002, 2007]))
].copy()

# Keep only countries with data in both years
valid_countries = (
    asia.groupby('country')['year']
    .nunique()
)
valid_countries = valid_countries[valid_countries == 2].index

asia = asia[asia['country'].isin(valid_countries)]

# Create scatter plot
fig = px.scatter(
    asia,
    x='gdpPercap',
    y='lifeExp',
    color='year',
    hover_name='country',
    color_discrete_map={
        2002: '#3498DB',
        2007: '#E67E22'
    },
    title='Asia: GDP per Capita vs Life Expectancy (2002–2007)',
    labels={
        'gdpPercap': 'GDP per Capita',
        'lifeExp': 'Life Expectancy (Years)',
        'year': 'Year'
    }
)

# Connect each country's 2002 and 2007 positions
for country in valid_countries:
    country_data = asia[asia['country'] == country].sort_values('year')

    x1 = country_data.iloc[0]['gdpPercap']
    y1 = country_data.iloc[0]['lifeExp']
    x2 = country_data.iloc[1]['gdpPercap']
    y2 = country_data.iloc[1]['lifeExp']

    fig.add_annotation(
        x=x2,
        y=y2,
        ax=x1,
        ay=y1,
        xref='x',
        yref='y',
        axref='x',
        ayref='y',
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1,
        arrowcolor='gray'
    )

fig.update_layout(
    template='plotly_white',
    width=1000,
    height=650
)

fig.show()


## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [4]:

# Task 2 - Bubble Chart: Tell a Story

import pandas as pd
import plotly.graph_objects as go

# Select the full 2007 dataset
data_2007 = df[df['year'] == 2007].copy()

# Remove rows with missing or invalid values
data_2007 = data_2007.dropna(
    subset=['gdpPercap', 'lifeExp', 'pop', 'continent', 'country']
)

data_2007 = data_2007[
    (data_2007['gdpPercap'] > 0) &
    (data_2007['pop'] > 0)
]

# Highlight Asia and keep all other continents grey
asia = data_2007[data_2007['continent'] == 'Asia']
others = data_2007[data_2007['continent'] != 'Asia']

# Calculate bubble size reference
size_ref = 2.0 * data_2007['pop'].max() / (45 ** 2)

# Create bubble chart
fig = go.Figure()

# Other continents in grey
fig.add_trace(go.Scatter(
    x=others['gdpPercap'],
    y=others['lifeExp'],
    mode='markers',
    name='Other Continents',
    text=others['country'],
    marker=dict(
        size=others['pop'],
        sizemode='area',
        sizeref=size_ref,
        sizemin=4,
        color='#BDBDBD',
        opacity=0.65,
        line=dict(width=0.5, color='white')
    ),
    hovertemplate=
        '<b>%{text}</b><br>' +
        'GDP per Capita: %{x:,.0f}<br>' +
        'Life Expectancy: %{y:.1f} years<br>' +
        '<extra>Other Continents</extra>'
))

# Asia highlighted in blue
fig.add_trace(go.Scatter(
    x=asia['gdpPercap'],
    y=asia['lifeExp'],
    mode='markers',
    name='Asia',
    text=asia['country'],
    marker=dict(
        size=asia['pop'],
        sizemode='area',
        sizeref=size_ref,
        sizemin=4,
        color='#2980B9',
        opacity=0.8,
        line=dict(width=0.8, color='white')
    ),
    hovertemplate=
        '<b>%{text}</b><br>' +
        'GDP per Capita: %{x:,.0f}<br>' +
        'Life Expectancy: %{y:.1f} years<br>' +
        '<extra>Asia</extra>'
))

# Calculate Asia statistics for the insight
median_gdp = asia['gdpPercap'].median()
median_life = asia['lifeExp'].median()

# Add annotation explaining Asia's story
fig.add_annotation(
    x=0.98,
    y=0.05,
    xref='paper',
    yref='paper',
    text=(
        f"<b>Asia's story (2007)</b><br>"
        f"Median GDP per capita: ${median_gdp:,.0f}<br>"
        f"Median life expectancy: {median_life:.1f} years<br>"
        "Asian countries show different levels<br>"
        "of income and life expectancy."
    ),
    showarrow=False,
    align='right',
    bgcolor='white',
    bordercolor='#2980B9',
    borderwidth=1,
    borderpad=8
)

# Chart layout
fig.update_layout(
    title='Global Development in 2007: Asia Highlighted',
    xaxis=dict(
        title='GDP per Capita (Log Scale)',
        type='log'
    ),
    yaxis_title='Life Expectancy (Years)',
    template='plotly_white',
    width=1100,
    height=700,
    legend_title='Continent',
    hovermode='closest'
)

fig.show()
